In [1]:
import pandas as pd
import numpy as np
import torch
import transformers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModel, BertTokenizerFast, DistilBertTokenizerFast, AutoTokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.utils.data import Dataset
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm
from nltk.tokenize import sent_tokenize, word_tokenize
import os
import nltk
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import f1_score
import shap
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read Data

In [12]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
train_df = pd.read_csv('../data/combined_letters_degendered_with_topics_train_granular.csv')
val_df = pd.read_csv('../data/combined_letters_degendered_with_topics_val_granular.csv')
test_df = pd.read_csv('../data/combined_letters_degendered_with_topics_test_granular.csv')

# Create Training and Test Sets

In [13]:
bert = AutoModel.from_pretrained("distilbert/distilbert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [14]:
class TextWithTopicsDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.text_data = df['full_text'].tolist()
        self.topic_features = df.filter(like='topic_').values
        self.labels = df['label'].values
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.text_data)

    def __getitem__(self, idx):
        text = self.text_data[idx]
        topics = torch.tensor(self.topic_features[idx], dtype=torch.float)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        tokens = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'topic_feats': topics,
            'label': label
        }

In [15]:
train_dataset = TextWithTopicsDataset(train_df, tokenizer)
val_dataset = TextWithTopicsDataset(val_df, tokenizer)
test_dataset = TextWithTopicsDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

In [16]:
# freeze weights, uncomment if you would like to unfreeze weights
for param in bert.parameters():
    param.requires_grad = False


In [17]:
# Unfreeze top 2 layers to start (layers 4 and 5)
for i in [4, 5]:
    for param in bert.transformer.layer[i].parameters():
        param.requires_grad = True


# Create Model

In [18]:
class BERTWithTopics(nn.Module):
    def __init__(self, bert, topic_feat_dim, num_classes):
        super(BERTWithTopics, self).__init__()
        self.bert = bert
        self.dropout = nn.Dropout(0.1)
        self.relu = nn.ReLU()

        combined_dim = self.bert.config.hidden_size + topic_feat_dim  # 768 + number of topic features
        self.fc1 = nn.Linear(combined_dim, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_ids, attention_mask, topic_feats):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embed = bert_out.last_hidden_state[:, 0]  # [CLS] token

        x = torch.cat((cls_embed, topic_feats), dim=1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return self.softmax(x)

In [34]:
model = BERTWithTopics(bert, 251, 2)
model = model.to(device)

In [35]:
batch_size = 16
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_df['label']), y=train_df['label'])
weights= torch.tensor(class_weights,dtype=torch.float)
weights = weights.to(device)
cross_entropy  = nn.NLLLoss(weight=weights)
epochs = 10

In [36]:
# function to train the model
def train():

  model.train()

  total_loss, total_accuracy = 0, 0

  # empty list to save model predictions
  total_preds=[]
  total_labels=[]

  # iterate over batches
  for step,batch in enumerate(tqdm(train_loader, desc="Training", leave=True)):

    # # progress update after every 50 batches.
    # if step % 50 == 0 and not step == 0:
    #   print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(train_dataloader)))

    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    topic_feats = batch['topic_feats'].to(device)
    labels = batch['label'].to(device)

    # clear previously calculated gradients
    model.zero_grad()

    # get model predictions for the current batch
    preds = model(input_ids, attention_mask, topic_feats)

    # compute the loss between actual and predicted values
    loss = cross_entropy(preds, labels)

    # add on to the total loss
    total_loss = total_loss + loss.item()

    # backward pass to calculate the gradients
    loss.backward()

    # clip the the gradients to 1.0. It helps in preventing the exploding gradient problem
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    # update parameters
    optimizer.step()

    # model predictions are stored on GPU. So, push it to CPU
    preds=preds.detach().cpu().numpy()

    # append the model predictions
    total_preds.append(preds)

    labels=labels.detach().cpu().numpy()

    # append labels
    total_labels.append(labels)

  # compute the training loss of the epoch
  avg_loss = total_loss / len(train_loader)

  # predictions are in the form of (no. of batches, size of batch, no. of classes).
  # reshape the predictions in form of (number of samples, no. of classes)
  total_preds  = np.concatenate(total_preds, axis=0)

  total_labels = np.concatenate(total_labels, axis=0)

  epoch_preds = np.argmax(total_preds, axis = 1)

  print('Training Classification Report: \n', classification_report(total_labels, epoch_preds))
  print('Training Confusion Matrix: \n', confusion_matrix(total_labels, epoch_preds))

  #returns the loss and predictions
  return avg_loss, total_preds

In [37]:
# function for evaluating the model
def evaluate():

  print("\nEvaluating...")

  # deactivate dropout layers
  model.eval()

  total_loss, total_accuracy = 0, 0

  # empty list to save the model predictions
  total_preds = []
  total_labels = []

  # iterate over batches
  for step,batch in enumerate(val_loader):

    # # Progress update every 50 batches.
    # if step % 50 == 0 and not step == 0:

    #   # Calculate elapsed time in minutes.
    #   elapsed = format_time(time.time() - t0)

    #   # Report progress.
    #   print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(val_dataloader)))

    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    topic_feats = batch['topic_feats'].to(device)
    labels = batch['label'].to(device)

    # deactivate autograd
    with torch.no_grad():

      # model predictions
      preds = model(input_ids, attention_mask, topic_feats)

      # compute the validation loss between actual and predicted values
      loss = cross_entropy(preds,labels)

      total_loss = total_loss + loss.item()

      preds = preds.detach().cpu().numpy()

      total_preds.append(preds)

      labels = labels.detach().cpu().numpy()

      total_labels.append(labels)

  # compute the validation loss of the epoch
  avg_loss = total_loss / len(val_loader)

  # reshape the predictions in form of (number of samples, no. of classes)
  total_preds  = np.concatenate(total_preds, axis=0)

  total_labels = np.concatenate(total_labels, axis=0)

  epoch_preds = np.argmax(total_preds, axis = 1)

  print('Validation Classification Report: \n', classification_report(total_labels, epoch_preds))
  print('Validation Confusion Matrix: \n', confusion_matrix(total_labels, epoch_preds))

  return avg_loss, (epoch_preds, total_labels)

In [38]:
# set initial loss to infinite
# best_valid_loss = float('inf')
best_macro_f1 = 0.0

# empty lists to store training and validation loss of each epoch
train_losses=[]
valid_losses=[]
macro_f1_scores = []

#for each epoch
for epoch in tqdm(range(epochs), desc="Training"):

    print('\n Epoch {:} / {:}'.format(epoch + 1, epochs))

    #train model
    train_loss, _ = train()

    #evaluate model
    valid_loss, (all_preds, all_labels) = evaluate()

    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    macro_f1_scores.append(macro_f1)

    # Save the best model based on F1 score
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        print('Model Saved (best F1)!')
        torch.save(model, '../saved_models/saved_model_with_topic_vector.pt')

    #save the best model
    # if valid_loss < best_valid_loss:
    #     best_valid_loss = valid_loss
    #     print('Model Saved!')
    #     torch.save(model, '../saved_models/saved_model.pt')

    # append training and validation loss
    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f'\nTraining Loss: {train_loss:.3f}')
    print(f'Validation Loss: {valid_loss:.3f}')

Training:   0%|          | 0/10 [00:00<?, ?it/s]


 Epoch 1 / 10


Training:   0%|          | 0/449 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.33      0.52      0.40      2229
           1       0.71      0.52      0.60      4953

    accuracy                           0.52      7182
   macro avg       0.52      0.52      0.50      7182
weighted avg       0.59      0.52      0.54      7182

Training Confusion Matrix: 
 [[1157 1072]
 [2357 2596]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.34      0.11      0.17       278
           1       0.69      0.90      0.79       621

    accuracy                           0.66       899
   macro avg       0.52      0.51      0.48       899
weighted avg       0.59      0.66      0.60       899

Validation Confusion Matrix: 
 [[ 31 247]
 [ 59 562]]
Model Saved (best F1)!

Training Loss: 0.693
Validation Loss: 0.691

 Epoch 2 / 10


Training:   0%|          | 0/449 [00:00<?, ?it/s]

...

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.52      0.69      0.60      2229
           1       0.84      0.71      0.77      4953

    accuracy                           0.71      7182
   macro avg       0.68      0.70      0.68      7182
weighted avg       0.74      0.71      0.72      7182

Training Confusion Matrix: 
 [[1548  681]
 [1412 3541]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.38      0.54      0.45       278
           1       0.75      0.61      0.67       621

    accuracy                           0.59       899
   macro avg       0.56      0.57      0.56       899
weighted avg       0.63      0.59      0.60       899

Validation Confusion Matrix: 
 [[151 127]
 [245 376]]

Training Loss: 0.567
Validation Loss: 0.735

 Epoch 9 / 10


Training:   0%|          | 0/449 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.56      0.72      0.63      2229
           1       0.85      0.74      0.80      4953

    accuracy                           0.74      7182
   macro avg       0.71      0.73      0.71      7182
weighted avg       0.76      0.74      0.74      7182

Training Confusion Matrix: 
 [[1603  626]
 [1271 3682]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.40      0.41      0.41       278
           1       0.73      0.73      0.73       621

    accuracy                           0.63       899
   macro avg       0.57      0.57      0.57       899
weighted avg       0.63      0.63      0.63       899

Validation Confusion Matrix: 
 [[114 164]
 [170 451]]

Training Loss: 0.537
Validation Loss: 0.778

 Epoch 10 / 10


Training:   0%|          | 0/449 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.60      0.76      0.67      2229
           1       0.88      0.77      0.82      4953

    accuracy                           0.77      7182
   macro avg       0.74      0.76      0.74      7182
weighted avg       0.79      0.77      0.77      7182

Training Confusion Matrix: 
 [[1685  544]
 [1136 3817]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.36      0.54      0.43       278
           1       0.73      0.57      0.64       621

    accuracy                           0.56       899
   macro avg       0.55      0.55      0.53       899
weighted avg       0.62      0.56      0.57       899

Validation Confusion Matrix: 
 [[150 128]
 [269 352]]

Training Loss: 0.501
Validation Loss: 0.804


# Test Model

In [39]:
model = torch.load('../saved_models/saved_model_with_topic_vector.pt', weights_only=False)

In [40]:

model.eval()  # Set model to eval mode

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        topic_feats = batch['topic_feats'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask, topic_feats)

        # Get predicted class (as indices)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [41]:
  print('Test Classification Report: \n', classification_report(all_labels, all_preds))
  print('Test Confusion Matrix: \n', confusion_matrix(all_labels, all_preds))

Test Classification Report: 
               precision    recall  f1-score   support

           0       0.41      0.44      0.43       279
           1       0.74      0.72      0.73       620

    accuracy                           0.63       899
   macro avg       0.58      0.58      0.58       899
weighted avg       0.64      0.63      0.64       899

Test Confusion Matrix: 
 [[124 155]
 [175 445]]
